# 5.5.读写文件

到目前为止，我们讨论了如何处理数据，以及如何构建、训练和测试深度学习模型。然而，有时我们希望保存训练的模型，以备将来在各种环境中使用（比如在部署中进行预测）。此外，当运行一个耗时较长的训练过程时，最佳的做法是定期保存中间结果，以确保在服务器电源被不小心断掉时，我们不会损失几天的计算结果。因此，现在是时候学习如何加载和存储权重向量和整个模型了。

---

## 环境准备

In [ ]:
%pip install pypto==0.2.0 torch torch_npu numpy

In [ ]:
import os
os.environ["TILE_FWK_DEVICE_ID"] = "0"

import pypto
import torch
import torch_npu
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import math

from src.pto_layers import PyPTOLinear, PyPTOReLU

---

## 5.5.1.加载和保存张量

对于单个张量，我们可以直接调用`load`和`save`函数分别读写它们。这两个函数都要求我们提供一个名称，`save`要求将要保存的变量作为输入。

In [4]:
x = torch.arange(4).npu()
torch.save(x, 'x-file')

我们现在可以将存储在文件中的数据读回内存。


In [6]:
x2 = torch.load('x-file', weights_only=False)
x2

tensor([0, 1, 2, 3], device='npu:0')

<details class="code-note" style="border: 1px solid #e3e3ee; border-radius: 4px; margin: 20px 0; overflow: hidden;">
  <summary style="padding: 10px 14px; font-weight: 500; cursor: pointer; background-color: #f8f8fa; list-style: none; color: #374151; font-size: 14px; letter-spacing: 0.01em;">点击：查看/折叠代码说明</summary>
  <div class="code-note-content" style="padding: 14px; line-height: 1.65; color: #4b5563; font-size: 14px; background-color: #ffffff;">
    <ul style="margin: 0; padding-left: 20px;">
      <li style="margin: 0 0 8px 0;">新版 PyTorch 将<code style="font-family: Consolas, monospace; background-color: #f3f4f6; padding: 2px 5px; border-radius: 3px; color: #1f2937; font-size: 13px;">torch.load()</code>的默认行为调整为<code style="font-family: Consolas, monospace; background-color: #f3f4f6; padding: 2px 5px; border-radius: 3px; color: #1f2937; font-size: 13px;">weights_only=True</code>。对于保存在 NPU 上的 Tensor，加载时除了恢复 Tensor 数据外，还需要恢复其设备信息（如<code style="font-family: Consolas, monospace; background-color: #f3f4f6; padding: 2px 5px; border-radius: 3px; color: #1f2937; font-size: 13px;">device='npu:0'</code>）以及相关的 NPU 存储状态，这些内容不属于<code style="font-family: Consolas, monospace; background-color: #f3f4f6; padding: 2px 5px; border-radius: 3px; color: #1f2937; font-size: 13px;">weights_only=True</code>的默认支持范围，因此需要显式指定<code style="font-family: Consolas, monospace; background-color: #f3f4f6; padding: 2px 5px; border-radius: 3px; color: #1f2937; font-size: 13px;">weights_only=True</code>。</li>
    </ul>
  </div>
</details>

<details class="original-text" style="border: 1px solid #e3e3ee; border-radius: 4px; margin: 20px 0; overflow: hidden;">
  <summary style="padding: 10px 14px; font-weight: 500; cursor: pointer; background-color: #f8f8fa; list-style: none; color: #374151; font-size: 14px; letter-spacing: 0.01em;">点击：查看/折叠原文</summary>
  <div class="original-content" style="padding: 14px; background-color: #ffffff; line-height: 1.65; color: #4b5563; font-size: 14px;">
    <pre style="background-color: transparent; color: #1f2937; border: 1px solid #e5e7eb; border-radius: 0; font-size: 13px; padding: 12px; margin: 0 0 10px 0; font-family: Consolas, monospace;">x2 = torch.load('x-file')
x2</pre>
    <span style="color: #1f2937; font-family: Consolas, monospace; font-size: 13px;">tensor([0, 1, 2, 3])</span>
  </div>
</details>

我们可以**存储一个张量列表，然后把它们读回内存。**


In [7]:
y = torch.zeros(4).npu()
torch.save([x, y],'x-files')
x2, y2 = torch.load('x-files', weights_only=False)
(x2, y2)

(tensor([0, 1, 2, 3], device='npu:0'),
 tensor([0., 0., 0., 0.], device='npu:0'))

我们甚至可以**写入或读取从字符串映射到张量的字典**。当我们要读取或写入模型中的所有权重时，这很方便。


In [8]:
mydict = {'x': x, 'y': y}
torch.save(mydict, 'mydict')
mydict2 = torch.load('mydict', weights_only=False)
mydict2

{'x': tensor([0, 1, 2, 3], device='npu:0'),
 'y': tensor([0., 0., 0., 0.], device='npu:0')}

---

## 5.5.2.加载和保存模型参数

保存单个权重向量（或其他张量）确实有用，但是如果我们想保存整个模型，并在以后加载它们，单独保存每个向量则会变得很麻烦。毕竟，我们可能有数百个参数散布在各处。因此，深度学习框架提供了内置函数来保存和加载整个网络。需要注意的一个重要细节是，这将保存模型的参数而不是保存整个模型。例如，如果我们有一个3层多层感知机，我们需要单独指定架构。因为模型本身可以包含任意代码，所以模型本身难以序列化。因此，为了恢复模型，我们需要用代码生成架构，然后从磁盘加载参数。让我们从熟悉的多层感知机开始尝试一下。


In [9]:
layer = PyPTOLinear(20, 256)

x = torch.randn(2,20).npu()

y = layer(x)

print(y.shape)

torch.Size([2, 256])


In [10]:
layer = PyPTOLinear(256, 10)

x = torch.randn(2,256).npu()

y = layer(x)

torch.npu.synchronize()

print(y.shape)

torch.Size([2, 10])


In [11]:
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.hidden = PyPTOLinear(20, 256)
        self.output = PyPTOLinear(256, 10)

    def forward(self, x):
        return self.output(F.relu(self.hidden(x)))

net = MLP()
X = torch.randn(size=(2, 20)).npu()
Y = net(X)

<details class="original-text" style="border: 1px solid #e3e3ee; border-radius: 4px; margin: 20px 0; overflow: hidden;">
  <summary style="padding: 10px 14px; font-weight: 500; cursor: pointer; background-color: #f8f8fa; list-style: none; color: #374151; font-size: 14px; letter-spacing: 0.01em;">点击：查看/折叠原文</summary>
  <div class="original-content" style="padding: 14px; background-color: #ffffff; line-height: 1.65; color: #4b5563; font-size: 14px;">
<pre style="background-color: transparent; color: #1f2937; border: 1px solid #e5e7eb; border-radius: 0; font-size: 13px; padding: 12px; margin: 0; font-family: Consolas, monospace;">class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.hidden = nn.Linear(20, 256)
        self.output = nn.Linear(256, 10)
    def forward(self, x):
        return self.output(F.relu(self.hidden(x)))

net = MLP()
X = torch.randn(size=(2, 20))
Y = net(X)</pre>
  </div>
</details>

接下来，我们**将模型的参数存储在一个叫做“mlp.params”的文件中。**


In [12]:
torch.save(net.state_dict(), 'mlp.params')

为了恢复模型，我们**实例化了原始多层感知机模型的一个备份。**这里我们不需要随机初始化模型参数，而是**直接读取文件中存储的参数。**


In [14]:
clone = MLP()
clone.load_state_dict(torch.load('mlp.params', weights_only=False))
clone.eval()

MLP(
  (hidden): PyPTOLinear()
  (output): PyPTOLinear()
)

<details class="original-text" style="border: 1px solid #e3e3ee; border-radius: 4px; margin: 20px 0; overflow: hidden;">
  <summary style="padding: 10px 14px; font-weight: 500; cursor: pointer; background-color: #f8f8fa; list-style: none; color: #374151; font-size: 14px; letter-spacing: 0.01em;">点击：查看/折叠原文</summary>
  <div class="original-content" style="padding: 14px; background-color: #ffffff; line-height: 1.65; color: #4b5563; font-size: 14px;">
<pre style="background-color: transparent; color: #1f2937; border: 1px solid #e5e7eb; border-radius: 0; font-size: 13px; padding: 12px; margin: 0; font-family: Consolas, monospace;">clone = MLP()
clone.load_state_dict(torch.load'mlp.params')
clone.eval()</pre>
<pre style="color: #1f2937; font-family: Consolas, monospace; font-size: 13px;">MLP(
  (hidden): Linear(in_features=20, out_features=256, bias=True)
  (output): Linear(in_features=256, out_features=10, bias=True)
)</pre>
  </div>
</details>

由于两个实例具有相同的模型参数，在输入相同的`X`时，两个实例的计算结果应该相同。让我们来验证一下。


In [15]:
Y_clone = clone(X)
Y_clone == Y

tensor([[True, True, True, True, True, True, True, True, True, True],
        [True, True, True, True, True, True, True, True, True, True]],
       device='npu:0')

---
## 5.5.3.小结

* `save`和`load`函数可用于张量对象的文件读写。
* 我们可以通过参数字典保存和加载网络的全部参数。
* 保存架构必须在代码中完成，而不是在参数中完成。

---
## 5.5.4.练习

1. 即使不需要将经过训练的模型部署到不同的设备上，存储模型参数还有什么实际的好处？
1. 假设我们只想复用网络的一部分，以将其合并到不同的网络架构中。比如想在一个新的网络中使用之前网络的前两层，该怎么做？
1. 如何同时保存网络架构和参数？需要对架构加上什么限制？

详细参考答案见[05.05_reference_answer](./answers/05.05_reference_answer.ipynb)

### 参考答案（PyPTO）

In [ ]:
!cat ./answers/txt/05.05_reference_pypto.txt

### 参考答案（PyTorch）

In [ ]:
!cat ./answers/txt/05.05_reference_pytorch.txt